<a href="https://colab.research.google.com/github/Adityagujjar-08/dataviz-exercises-adityapatil/blob/main/lecture04_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Filter for Americas continent and the years 2002 and 2007
continent_choice = 'Americas'
df_task1 = df.loc[
    (df['continent'] == continent_choice) &
    (df['year'].isin([2002, 2007]))
].copy()

# Convert year to string so Plotly maps it to discrete colors
df_task1['year'] = df_task1['year'].astype(str)

# Create the scatter plot
fig_task1 = px.scatter(
    df_task1,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    hover_name='country',
    log_x=True,  # Apply logarithmic scale to GDP per capita
    title='Americas Progress (2002 vs 2007): Outward Shifts in Income and Lifespan',
    labels={
        'gdpPercap': 'GDP per Capita (USD, Log Scale)',
        'lifeExp': 'Life Expectancy (Years)',
        'year': 'Year'
    },
    template='plotly_white'
)

# Customize markers
fig_task1.update_traces(marker=dict(size=12, opacity=0.85, line=dict(width=1, color='DarkSlateGrey')))

fig_task1.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Filter for the year 2007
df_2007 = df.loc[df['year'] == 2007].copy()

# Create a highlight category (Asia vs Other)
highlight_continent = 'Asia'
df_2007['highlight_group'] = df_2007['continent'].apply(lambda x: x if x == highlight_continent else 'Other (Greyed)')

# Build custom color mapping adhering to the gray-and-highlight rule
color_map = {
    'Asia': '#1f77b4',          # Highlight color
    'Other (Greyed)': '#d3d3d3' # Background grey
}

# Sort DataFrame to draw highlighted points on top
df_2007 = df_2007.sort_values(by='highlight_group', ascending=False)

# Build the bubble chart
fig_task2 = px.scatter(
    df_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='highlight_group',
    color_discrete_map=color_map,
    hover_name='country',
    log_x=True,
    size_max=55,  # Control maximum bubble size limit
    title='Asia in 2007: High Demographic Mass and Broad Life Expectancy Disparity',
    labels={
        'gdpPercap': 'GDP per Capita (USD, Log Scale)',
        'lifeExp': 'Life Expectancy (Years)',
        'highlight_group': 'Continent',
        'pop': 'Population'
    },
    template='plotly_white'
)

# Customize markers
fig_task2.update_traces(marker=dict(opacity=0.75, line=dict(width=1, color='DimGray')))

# Add annotation describing the highlighted group's story
fig_task2.add_annotation(
    x=2500,  # X-coordinate around China & India in 2007
    y=67,
    text='China & India: Massive population bubbles located in the mid-income tier',
    showarrow=True,
    arrowhead=1,
    ax=0,
    ay=50,
    font=dict(size=11, color='black'),
    bgcolor='white',
    bordercolor='#1f77b4',
    borderwidth=1
)

fig_task2.show()